###### Imports and Settings

In [1]:
import pandas as pd
import numpy as np
import requests
#import io
import pickle
from collections import deque
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
def percent(x, y):
    return x/y

## Calculations

In [3]:
data = pd.read_csv('../../data/SF12010.csv')

## Population + Projections Summary: 

In [4]:
data['Population'] = data['agebysex_total_series']
data = data.drop(columns = ['agebysex_total_series','pop'])

## Age Sex Race Summary:

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [5]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']

#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Age:Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['Age:18 to 54'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:55 and Older'] = sum(fifty5pluslist)

#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:65 and Older'] = sum(sixty5pluslist)

cols = ['age_total_male','age_total_female','age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [6]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_total_series','raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino']
data = data.drop(columns = cols)

## Household Summary:  

### Households by Household Type  
+ Total Households  
+ Family Households  
+ Family Households: Married-Couple Family  
+ Family Households: Other Family  
+ Family Households: Other Family: Male Householder no Wife Present  
+ Family Households: Other Family: Female Householder no Husband Present  
+ Nonfamily Households  
+ Nonfamily Household: Male Householder  
+ Nonfamily Household: Female Householder  

*They don't have the nonfamily male or female - replaced by Householder alone or not alone*

In [7]:
data['Total Households'] = data['hhtype_total_series']
data['Family Households'] = data['hhtype_familyhh']
data['Family Households: Married Couple Family'] = data['hhtype_familyhh_marriedcouplefam']
data['Family Households: Not Married Couple Family'] = data['hhtype_familyhh_otherfam']
data['Family Households: Not Married Couple: Male no Spouse'] = data['hhtype_familyhh_malenospouse']
data['Family Households: Not Married Couple: Female no Spouse'] = data['hhtype_familyhh_femalenospouse']
data['Nonfamily Households'] = data['hhtype_nonfamhh']
data['Nonfamily Households: Householder Alone'] = data['hhtype_nonfamhh_householderalone']
data['Nonfamily Households: Householder not Alone'] = data['hhtype_nonfamhh_householdernotalone']

cols = ['hhtype_total_series','hhtype_familyhh','hhtype_familyhh_marriedcouplefam','hhtype_familyhh_otherfam','hhtype_familyhh_malenospouse',
        'hhtype_familyhh_femalenospouse','hhtype_nonfamhh','hhtype_nonfamhh_householderalone','hhtype_nonfamhh_householdernotalone']
data = data.drop(columns = cols)

### Average Household Size 
Median - drop for incorporated and unincorporated

In [8]:
data['Average Household Size'] = data['hhsize_avg']
data = data.drop(columns = ['hhsize_avg'])

### Occupancy Status, and Percentages  
+ Occupancy Total Households
+ Occupied  
+ Vacant  

In [9]:
data['Occupancy:Total Households'] = data['occupancy_total_series']
data['Occupancy:Occupied Units'] = data['occupancy_occupiedunits']
data['Occupancy%:Occupied Units'] = percent(data['Occupancy:Occupied Units'], data['Occupancy:Total Households'])
data['Occupancy:Vacant Units'] = data['occupancy_vacantunits']
data['Occupancy%:Vacant Units'] = percent(data['Occupancy:Vacant Units'], data['Occupancy:Total Households'])

cols = ['occupancy_total_series','occupancy_occupiedunits','occupancy_vacantunits']
data = data.drop(columns = cols)

### Tenure, and Percentages  
+ Tenure Total Households  
+ Owners  
+ Renters

In [10]:
data['Tenure:Total Households'] = data['tenure_total_series']
data['Tenure:Owners'] = data['tenure_owneroccunits_mortgage']+data['tenure_owneroccunits_nomortgage']
data['Tenure%:Owners'] = percent(data['Tenure:Owners'], data['Tenure:Total Households'])
data['Tenure:Renters'] = data['tenure_renteroccunits']
data['Tenure%:Renters'] = percent(data['Tenure:Renters'], data['Tenure:Total Households'])
cols = ['units_allhousing','tenure_total_series','tenure_owneroccunits_mortgage','tenure_owneroccunits_nomortgage','tenure_renteroccunits']
data = data.drop(columns = cols)

In [11]:
data = data.drop(columns = ['StateFIPS', 'GeoFIPS'])

In [12]:
data.head()

,NAME,Population,Male Under 5,Female Under 5,Male 5 to 9,Female 5 to 9,Male 10 to 14,Female 10 to 14,Male 15 to 17,Female 15 to 17,Male 18 to 24,Female 18 to 24,Male 25 to 34,Female 25 to 34,Male 35 to 44,Female 35 to 44,Male 45 to 54,Female 45 to 54,Male 55 to 64,Female 55 to 64,Male 65 to 74,Female 65 to 74,Male 75 to 84,Female 75 to 84,Male 85 and Older,Female 85 and Older,Age:Under 18,Age:18 to 54,Age:55 and Older,Age:65 and Older,White Alone,White Alone %,Black or African American Alone,Black or African American Alone %,American Indian Alaska Native Alone,American Indian Alaska Native Alone %,Asian Alone,Asian Alone %,Native Hawaiian Other Pacific Islander Alone,Native Hawaiian Other Pacific Islander Alone %,Some Other Race Alone,Some Other Race Alone %,Two or More Races,Two or More Races %,Hispanic or Latino,Hispanic or Latino %,Minority,Minority %,Total Households,Family Households,Family Households: Married Couple Family,Family Households: Not Married Couple Family,Family Households: Not Married Couple: Male no Spouse,Family Households: Not Married Couple: Female no Spouse,Nonfamily Households,Nonfamily Households: Householder Alone,Nonfamily Households: Householder not Alone,Average Household Size,Occupancy:Total Households,Occupancy:Occupied Units,Occupancy%:Occupied Units,Occupancy:Vacant Units,Occupancy%:Vacant Units,Tenure:Total Households,Tenure:Owners,Tenure%:Owners,Tenure:Renters,Tenure%:Renters
0,"Williamson County, Tennessee",183182.0,6324.0,6039.0,7997.0,7718.0,8372.0,7870.0,4712.0,4597.0,5558.0,5028.0,8437.0,9735.0,14063.0,15361.0,15079.0,16388.0,10934.0,11163.0,5028.0,5457.0,2190.0,3045.0,642.0,1445.0,53629.0,89649.0,39904.0,17807.0,163728.0,0.893800,7941.0,0.043350,396.0,0.002162,5517.0,0.030118,82.0,0.000448,2784.0,0.015198,2734.0,0.014925,8166.0,0.044579,24413.0,0.133272,64886.0,51242.0,44172.0,7070.0,1734.0,5336.0,13644.0,11505.0,2139.0,2.81,68498.0,64886.0,0.947269,3612.0,0.052731,64886.0,52717.0,0.812456,12169.0,0.187544
1,"Wilson County, Tennessee",113993.0,3746.0,3572.0,4118.0,3933.0,4271.0,4086.0,2473.0,2362.0,4288.0,4009.0,6254.0,6853.0,8370.0,8784.0,9036.0,9536.0,7037.0,7397.0,4199.0,4419.0,1627.0,2278.0,415.0,930.0,28561.0,57130.0,28302.0,13868.0,101379.0,0.889344,7297.0,0.064013,395.0,0.003465,1276.0,0.011194,44.0,0.000386,1751.0,0.015361,1851.0,0.016238,3691.0,0.032379,14276.0,0.125236,42563.0,32177.0,25571.0,6606.0,1845.0,4761.0,10386.0,8452.0,1934.0,2.65,45568.0,42563.0,0.934055,3005.0,0.065945,42563.0,33730.0,0.792472,8833.0,0.207528
2,"Houston County, Tennessee",8426.0,245.0,228.0,304.0,276.0,287.0,298.0,186.0,168.0,330.0,315.0,442.0,425.0,506.0,544.0,599.0,588.0,568.0,614.0,430.0,440.0,203.0,269.0,44.0,117.0,1992.0,3749.0,2685.0,1503.0,8015.0,0.951222,193.0,0.022905,24.0,0.002848,25.0,0.002967,3.0,0.000356,35.0,0.004154,131.0,0.015547,129.0,0.015310,495.0,0.058747,3349.0,2285.0,1752.0,533.0,165.0,368.0,1064.0,942.0,122.0,2.46,4188.0,3349.0,0.799666,839.0,0.200334,3349.0,2544.0,0.759630,805.0,0.240370
3,"Humphreys County, Tennessee",18538.0,539.0,502.0,569.0,613.0,627.0,596.0,460.0,387.0,727.0,649.0,965.0,992.0,1171.0,1197.0,1347.0,1374.0,1257.0,1361.0,934.0,949.0,424.0,581.0,95.0,222.0,4293.0,8422.0,5823.0,3205.0,17629.0,0.950966,463.0,0.024976,82.0,0.004423,35.0,0.001888,6.0,0.000324,101.0,0.005448,222.0,0.011975,278.0,0.014996,1040.0,0.056101,7454.0,5116.0,3887.0,1229.0,369.0,860.0,2338.0,1969.0,369.0,2.46,8865.0,7454.0,0.840835,1411.0,0.159165,7454.0,5623.0,0.754360,1831.0,0.245640
4,"Maury County, Tennessee",80956.0,3021.0,2789.0,2702.0,2697.0,2699.0,2610.0,1668.0,1471.0,3346.0,3315.0,5159.0,5575.0,5086.0,5204.0,5945.0,6579.0,5124.0,5487.0,2735.0,3123.0,1313.0,2078.0,357.0,873.0,19657.0,40209.0,21090.0,10479.0,66677.0,0.823620,10154.0,0.125426,265.0,0.003273,489.0,0.006040,29.0,0.000358,1760.0,0.021740,1582.0,0.019541,3909.0,0.048285,16037.0,0.198095,31663.0,22449.0,16490.0,5959.0,1495.0,4464.0,9214.0,7705.0,1509.0,2.52,35254.0,31663.0,0.898139,3591.0,0.101861,31663.0,22813.0,0.72049

In [13]:
data = data.set_index('NAME')
data = data.add_suffix(' 2010')
data = data.reset_index()
data.head()

,NAME,Population 2010,Male Under 5 2010,Female Under 5 2010,Male 5 to 9 2010,Female 5 to 9 2010,Male 10 to 14 2010,Female 10 to 14 2010,Male 15 to 17 2010,Female 15 to 17 2010,Male 18 to 24 2010,Female 18 to 24 2010,Male 25 to 34 2010,Female 25 to 34 2010,Male 35 to 44 2010,Female 35 to 44 2010,Male 45 to 54 2010,Female 45 to 54 2010,Male 55 to 64 2010,Female 55 to 64 2010,Male 65 to 74 2010,Female 65 to 74 2010,Male 75 to 84 2010,Female 75 to 84 2010,Male 85 and Older 2010,Female 85 and Older 2010,Age:Under 18 2010,Age:18 to 54 2010,Age:55 and Older 2010,Age:65 and Older 2010,White Alone 2010,White Alone % 2010,Black or African American Alone 2010,Black or African American Alone % 2010,American Indian Alaska Native Alone 2010,American Indian Alaska Native Alone % 2010,Asian Alone 2010,Asian Alone % 2010,Native Hawaiian Other Pacific Islander Alone 2010,Native Hawaiian Other Pacific Islander Alone % 2010,Some Other Race Alone 2010,Some Other Race Alone % 2010,Two or More Races 2010,Two or More Races % 2010,Hispanic or Latino 2010,Hispanic or Latino % 2010,Minority 2010,Minority % 2010,Total Households 2010,Family Households 2010,Family Households: Married Couple Family 2010,Family Households: Not Married Couple Family 2010,Family Households: Not Married Couple: Male no Spouse 2010,Family Households: Not Married Couple: Female no Spouse 2010,Nonfamily Households 2010,Nonfamily Households: Householder Alone 2010,Nonfamily Households: Householder not Alone 2010,Average Household Size 2010,Occupancy:Total Households 2010,Occupancy:Occupied Units 2010,Occupancy%:Occupied Units 2010,Occupancy:Vacant Units 2010,Occupancy%:Vacant Units 2010,Tenure:Total Households 2010,Tenure:Owners 2010,Tenure%:Owners 2010,Tenure:Renters 2010,Tenure%:Renters 2010
0,"Williamson County, Tennessee",183182.0,6324.0,6039.0,7997.0,7718.0,8372.0,7870.0,4712.0,4597.0,5558.0,5028.0,8437.0,9735.0,14063.0,15361.0,15079.0,16388.0,10934.0,11163.0,5028.0,5457.0,2190.0,3045.0,642.0,1445.0,53629.0,89649.0,39904.0,17807.0,163728.0,0.893800,7941.0,0.043350,396.0,0.002162,5517.0,0.030118,82.0,0.000448,2784.0,0.015198,2734.0,0.014925,8166.0,0.044579,24413.0,0.133272,64886.0,51242.0,44172.0,7070.0,1734.0,5336.0,13644.0,11505.0,2139.0,2.81,68498.0,64886.0,0.947269,3612.0,0.052731,64886.0,52717.0,0.812456,12169.0,0.187544
1,"Wilson County, Tennessee",113993.0,3746.0,3572.0,4118.0,3933.0,4271.0,4086.0,2473.0,2362.0,4288.0,4009.0,6254.0,6853.0,8370.0,8784.0,9036.0,9536.0,7037.0,7397.0,4199.0,4419.0,1627.0,2278.0,415.0,930.0,28561.0,57130.0,28302.0,13868.0,101379.0,0.889344,7297.0,0.064013,395.0,0.003465,1276.0,0.011194,44.0,0.000386,1751.0,0.015361,1851.0,0.016238,3691.0,0.032379,14276.0,0.125236,42563.0,32177.0,25571.0,6606.0,1845.0,4761.0,10386.0,8452.0,1934.0,2.65,45568.0,42563.0,0.934055,3005.0,0.065945,42563.0,33730.0,0.792472,8833.0,0.207528
2,"Houston County, Tennessee",8426.0,245.0,228.0,304.0,276.0,287.0,298.0,186.0,168.0,330.0,315.0,442.0,425.0,506.0,544.0,599.0,588.0,568.0,614.0,430.0,440.0,203.0,269.0,44.0,117.0,1992.0,3749.0,2685.0,1503.0,8015.0,0.951222,193.0,0.022905,24.0,0.002848,25.0,0.002967,3.0,0.000356,35.0,0.004154,131.0,0.015547,129.0,0.015310,495.0,0.058747,3349.0,2285.0,1752.0,533.0,165.0,368.0,1064.0,942.0,122.0,2.46,4188.0,3349.0,0.799666,839.0,0.200334,3349.0,2544.0,0.759630,805.0,0.240370
3,"Humphreys County, Tennessee",18538.0,539.0,502.0,569.0,613.0,627.0,596.0,460.0,387.0,727.0,649.0,965.0,992.0,1171.0,1197.0,1347.0,1374.0,1257.0,1361.0,934.0,949.0,424.0,581.0,95.0,222.0,4293.0,8422.0,5823.0,3205.0,17629.0,0.950966,463.0,0.024976,82.0,0.004423,35.0,0.001888,6.0,0.000324,101.0,0.005448,222.0,0.011975,278.0,0.014996,1040.0,0.056101,7454.0,5116.0,3887.0,1229.0,369.0,860.0,2338.0,1969.0,369.0,2.46,8865.0,7454.0,0.840835,1411.0,0.159165,7454.0,5623.0,0.754360,1831.0,0.245640
4,"Maury County, Tennessee",80956.0,3021.0,2789.0,2702.0,2697.0,2699.0,2610.0,1668.0,1471.0,3346.0,3315.0,5159.0,5575.0,5086.0,5204.0,5945.0,6579.0,5124.0,5487.0,2735.0,3123.0,1

In [14]:
data.tail()

,NAME,Population 2010,Male Under 5 2010,Female Under 5 2010,Male 5 to 9 2010,Female 5 to 9 2010,Male 10 to 14 2010,Female 10 to 14 2010,Male 15 to 17 2010,Female 15 to 17 2010,Male 18 to 24 2010,Female 18 to 24 2010,Male 25 to 34 2010,Female 25 to 34 2010,Male 35 to 44 2010,Female 35 to 44 2010,Male 45 to 54 2010,Female 45 to 54 2010,Male 55 to 64 2010,Female 55 to 64 2010,Male 65 to 74 2010,Female 65 to 74 2010,Male 75 to 84 2010,Female 75 to 84 2010,Male 85 and Older 2010,Female 85 and Older 2010,Age:Under 18 2010,Age:18 to 54 2010,Age:55 and Older 2010,Age:65 and Older 2010,White Alone 2010,White Alone % 2010,Black or African American Alone 2010,Black or African American Alone % 2010,American Indian Alaska Native Alone 2010,American Indian Alaska Native Alone % 2010,Asian Alone 2010,Asian Alone % 2010,Native Hawaiian Other Pacific Islander Alone 2010,Native Hawaiian Other Pacific Islander Alone % 2010,Some Other Race Alone 2010,Some Other Race Alone % 2010,Two or More Races 2010,Two or More Races % 2010,Hispanic or Latino 2010,Hispanic or Latino % 2010,Minority 2010,Minority % 2010,Total Households 2010,Family Households 2010,Family Households: Married Couple Family 2010,Family Households: Not Married Couple Family 2010,Family Households: Not Married Couple: Male no Spouse 2010,Family Households: Not Married Couple: Female no Spouse 2010,Nonfamily Households 2010,Nonfamily Households: Householder Alone 2010,Nonfamily Households: Householder not Alone 2010,Average Household Size 2010,Occupancy:Total Households 2010,Occupancy:Occupied Units 2010,Occupancy%:Occupied Units 2010,Occupancy:Vacant Units 2010,Occupancy%:Vacant Units 2010,Tenure:Total Households 2010,Tenure:Owners 2010,Tenure%:Owners 2010,Tenure:Renters 2010,Tenure%:Renters 2010
76,Dickson Unincorporated,28646.0,894.0,869.0,1025.0,919.0,1075.0,962.0,710.0,635.0,1086.0,980.0,1536.0,1603.0,2054.0,2150.0,2415.0,2421.0,1828.0,1840.0,1175.0,1134.0,471.0,569.0,107.0,188.0,7089.0,14245.0,7312.0,3644.0,27222.0,0.950290,624.0,0.021783,95.0,0.003316,93.0,0.003247,1.0,0.000035,175.0,0.006109,436.0,0.015220,571.0,0.019933,1747.0,0.060986,10759.0,8164.0,6481.0,1683.0,552.0,1131.0,2595.0,2125.0,470.0,-12.59,11695.0,10759.0,0.919966,936.0,0.080034,10759.0,8954.0,0.832233,1805.0,0.167767
77,Humphreys Incorporated,7806.0,246.0,222.0,257.0,285.0,262.0,251.0,165.0,161.0,287.0,301.0,445.0,456.0,478.0,499.0,483.0,540.0,442.0,542.0,359.0,412.0,197.0,323.0,59.0,134.0,1849.0,3489.0,2468.0,1484.0,7245.0,0.928132,343.0,0.043941,35.0,0.004484,18.0,0.002306,1.0,0.000128,56.0,0.007174,108.0,0.013836,152.0,0.019472,626.0,0.080195,3167.0,2132.0,1525.0,607.0,163.0,444.0,1035.0,906.0,129.0,7.35,3520.0,3167.0,0.899716,353.0,0.100284,3167.0,2171.0,0.685507,996.0,0.314493
78,Humphreys Unincorporated,10732.0,293.0,280.0,312.0,328.0,365.0,345.0,295.0,226.0,440.0,348.0,520.0,536.0,693.0,698.0,864.0,834.0,815.0,819.0,575.0,537.0,227.0,258.0,36.0,88.0,2444.0,4933.0,3355.0,1721.0,10384.0,0.967574,120.0,0.011182,47.0,0.004379,17.0,0.001584,5.0,0.000466,45.0,0.004193,114.0,0.010622,126.0,0.011741,414.0,0.038576,4287.0,2984.0,2362.0,622.0,206.0,416.0,1303.0,1063.0,240.0,-4.89,5345.0,4287.0,0.802058,1058.0,0.197942,4287.0,3452.0,0.805225,835.0,0.194775
79,Montgomery Incorporated,132929.0,6505.0,6241.0,5296.0,5144.0,4677.0,4553.0,2757.0,2643.0,8747.0,9423.0,12590.0,12458.0,8400.0,8835.0,7105.0,7854.0,4649.0,5377.0,2448.0,3174.0,1305.0,1764.0,289.0,695.0,37816.0,75412.0,19701.0,9675.0,87135.0,0.655500,30798.0,0.231688,807.0,0.006071,3093.0,0.023268,628.0,0.004724,3747.0,0.028188,6721.0,0.050561,12302.0,0.092546,51764.0,0.389411,49439.0,34478.0,24135.0,10343.0,2286.0,8057.0,14961.0,11731.0,3230.0,2.63,54815.0,49439.0,0.901925,5376.0,0.098075,49439.0,27732.0,0.560934,21707.0,0.439066
80,Montgomery Unincorporated,39402.0,1298.0,1285.0,1458.0,1404.0,1580.0,1474.0,940.0,959.0,1833.0,1455.0,2308.0,2338.0,2978.0,3076.0,3113.0,3234.0,2268.0,2285.0,1226.0,1333.0,569.0,661.0,115.0,212.0,10398.0,20335.0,8669.0,4116.0,35201.

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 68 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   NAME                                                          81 non-null     object 
 1   Population 2010                                               81 non-null     float64
 2   Male Under 5 2010                                             81 non-null     float64
 3   Female Under 5 2010                                           81 non-null     float64
 4   Male 5 to 9 2010                                              81 non-null     float64
 5   Female 5 to 9 2010                                            81 non-null     float64
 6   Male 10 to 14 2010                                            81 non-null     float64
 7   Female 10 to 14 2010                                          81 non-null

In [16]:
data.to_csv('../../data/Prefix2010SF1.csv', index = False)